# 03 - Dendrogramas Avanzados, Estrategias de Corte y Clustermaps Bidireccionales

## Curso de Machine Learning: Modelos No Supervisados

Una vez construido el árbol jerárquico, debemos resolver el paso operativo decisivo: **¿Dónde y cómo trazamos la línea de corte para extraer los clusters finales?**

A diferencia de K-Means, donde cada corrida produce una sola partición plana, el dendrograma nos permite interactuar con la escala del problema:
- Podemos realizar un **corte horizontal rígido a una distancia umbral $t$**, descubriendo cuántos grupos naturales emergen a ese nivel de disimilaridad.
- O fijar un **número deseado de clusters $K$** y permitir que el algoritmo descienda por el árbol hasta aislar las $K$ ramas más representativas.

Además, en este cuaderno abordaremos una de las técnicas de visualización más aclamadas en bioinformática, genómica y finanzas cuantitativas: el **Bi-clustering o Clustermap**, donde se aplica clustering jerárquico aglomerativo **simultáneamente a las filas (observaciones) y a las columnas (variables)**.

---

### Contenidos
1. Estrategias de Corte en SciPy con `fcluster`: Criterio `'distance'` vs. `'maxclust'`
2. Identificación Visual del Corte Óptimo (Análisis de Saltos Verticales / Inconsistencia)
3. Manejo de Datasets Grandes: Dendrogramas Truncados (`truncate_mode='lastp'`)
4. Clustermaps Bidireccionales con Seaborn (`sns.clustermap`)
5. Interpretación de Patrones Cruzados (Bloques de Observaciones y Firmas de Variables)
6. Conclusiones


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.spatial.distance import pdist
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster, inconsistent
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

# Configuración visual
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
np.random.seed(42)


## 1. Estrategias de Corte: `fcluster` en SciPy

La función `scipy.cluster.hierarchy.fcluster` aplana la jerarquía de la matriz de enlace `Z` para devolver un vector de etiquetas enteras:

### Opción A: Corte por Distancia Umbral (`criterion='distance'`)
- Se traza una recta horizontal a la altura $t$. Todas las ramas que se unan por encima de $t$ permanecen como clusters separados.
- **Ventaja**: No fuerza un número arbitrario de grupos; respeta la escala métrica.

### Opción B: Corte por Número Máximo de Clusters (`criterion='maxclust'`)
- Se especifica $K$ y el algoritmo encuentra automáticamente la altura adecuada para que existan exactamente $K$ clusters.
- **Ventaja**: Garantiza el número exacto de grupos requerido para un caso de uso comercial o de asignación.


In [ ]:
# Carga del dataset y cálculo del enlace de Ward
wine = load_wine(as_frame=True)
X_scaled = StandardScaler().fit_transform(wine.data)
Z_ward = linkage(X_scaled, method='ward', metric='euclidean')

# 1. Corte por distancia fija t = 13.0
labels_dist = fcluster(Z_ward, t=13.0, criterion='distance')

# 2. Corte por maxclust t = 3
labels_k3 = fcluster(Z_ward, t=3, criterion='maxclust')

print(f'Clusters encontrados cortando a t = 13.0: {len(np.unique(labels_dist))}')
print(f'Distribución de muestras por cluster (corte por distancia):', pd.Series(labels_dist).value_counts().to_dict())
print(f'Clusters encontrados cortando a K = 3:     {len(np.unique(labels_k3))}')


## 2. Dónde Cortar: Saltos Verticales e Inconsistencia

La regla heurística visual para elegir la altura de corte consiste en buscar las **líneas verticales más largas en el dendrograma** que no sean intersectadas por ninguna línea horizontal.

Una línea vertical larga indica que los clusters fusionados estaban muy bien separados y tuvieron que "viajar" una gran distancia de disimilaridad antes de verse forzados a unirse.

Visualicemos el dendrograma completo destacando la línea de corte horizontal.


In [ ]:
plt.figure(figsize=(14, 6))
umbral_corte = 13.0

# Graficamos el dendrograma truncado a los últimos 30 nodos para claridad visual
dend = dendrogram(
    Z_ward, 
    p=30, 
    truncate_mode='lastp', 
    color_threshold=umbral_corte, 
    above_threshold_color='gray',
    leaf_rotation=90,
    leaf_font_size=10
)
plt.axhline(umbral_corte, color='crimson', linestyle='--', linewidth=2.5, label=f'Línea de Corte (t = {umbral_corte})')
plt.title('Dendrograma de Ward con Línea de Corte Horizontal Óptima (K=3 Clusters)', fontsize=13, fontweight='bold')
plt.ylabel('Distancia de Disimilaridad de Ward', fontsize=11, fontweight='bold')
plt.xlabel('Subgrupos y Conteo de Muestras Agrupadas', fontsize=11, fontweight='bold')
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()


## 3. Coeficiente de Inconsistencia (`inconsistent`)

SciPy permite cuantificar qué tan atípica es cada fusión en comparación con las fusiones previas de sus ramas secundarias utilizando `scipy.cluster.hierarchy.inconsistent`:

$$Y_{k} = \frac{Z[k, 2] - \bar{d}_{local}}{\sigma_{local}}$$

Un valor de inconsistencia alto ($Y_k > 1.5$) confirma que la unión ocurre a un salto de distancia inusualmente abrupto, señalando un límite natural de corte.


In [ ]:
# Cálculo de la matriz de inconsistencia (profundidad = 4)
incons_matrix = inconsistent(Z_ward, d=4)
df_incons = pd.DataFrame(incons_matrix, columns=['Media_Local', 'Desv_Local', 'N_Pasos_Locales', 'Coef_Inconsistencia'])
print('Últimos 8 pasos de fusión con sus coeficientes de inconsistencia:')
display(df_incons.tail(8).round(3))


## 4. Bi-Clustering y Mapas de Calor Jerárquicos: `sns.clustermap`

El **Clustermap** es una de las visualizaciones más poderosas de la analítica multivariada:
1. Aplica agrupamiento jerárquico aglomerativo a las **filas** (observaciones) y reordena las filas según el dendrograma lateral izquierdo.
2. Aplica simultáneamente agrupamiento jerárquico aglomerativo a las **columnas** (variables) y reordena las columnas según el dendrograma superior.
3. Pinta la matriz estandarizada resultante con un mapa de calor.

El resultado revela de inmediato **bloques cruzados de co-expresión**: conjuntos de observaciones que presentan valores simultáneamente altos o bajos en grupos específicos de variables.


In [ ]:
# Construcción de un Clustermap completo sobre el dataset Wine estandarizado
df_scaled = pd.DataFrame(X_scaled, columns=wine.feature_names)

# Paleta de colores para contrastar clusters descubiertos
colores_map = {1: '#1f77b4', 2: '#2ca02c', 3: '#d62728'}
colores_filas = pd.Series(labels_k3).map(colores_map)

g = sns.clustermap(
    df_scaled, 
    method='ward', 
    metric='euclidean',
    row_colors=colores_filas, 
    cmap='vlag', 
    center=0, 
    figsize=(13, 9),
    linewidths=0.0,
    cbar_kws={'label': 'Z-Score'}
)
g.fig.suptitle('Clustermap Bidireccional: Co-Agrupamiento Jerárquico de Muestras y Variables', fontsize=13, fontweight='bold', y=1.02)
plt.show()


## 5. Conclusiones

1. **`fcluster` con `distance` vs. `maxclust`**: La opción `distance` preserva el significado físico/geométrico de la escala, mientras que `maxclust` es idónea para satisfacer requerimientos de diseño de productos o portafolios fijos.
2. **Detección de Saltos**: Las ramas verticales más extensas del dendrograma marcan las divisiones más robustas del espacio.
3. **Clustermaps**: Permiten diagnosticar en una sola gráfica qué subgrupos de variables explican la separación de cada familia de observaciones.

En el siguiente cuaderno, **04_Matrices_de_Conectividad_y_Escalabilidad.ipynb**, aprenderemos cómo superar el cuello de botella computacional de $O(n^2)$ introduciendo grafos de conectividad espacial y de vecinos cercanos.
